In [3]:
import pandas as pd
import numpy as np


In [4]:
input_csv = "Crop_recommendation.csv"

df = pd.read_csv(input_csv)

print("Input shape:", df.shape)
df.head()


Input shape: (2200, 8)


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [5]:
# Soil temperature offset (°C)
SOIL_TEMP_OFFSET = {
    "Rice": 3.0,
    "Maize": 3.0,
    "Cotton": 4.0,
    "Banana": 3.5,
    "Apple": 5.0,
    "Mango": 4.5,
    "Orange": 4.0,
    "Sugarcane": 3.5,
    "Default": 3.5
}

# Soil moisture multiplier
SOIL_MOISTURE_FACTOR = {
    "Rice": 1.30,
    "Sugarcane": 1.25,
    "Banana": 1.15,
    "Cotton": 0.85,
    "Apple": 0.80,
    "Maize": 0.90,
    "Default": 1.00
}


In [6]:
np.random.seed(42)

def generate_soil_temperature(row):
    offset = SOIL_TEMP_OFFSET.get(row["label"], SOIL_TEMP_OFFSET["Default"])
    noise = np.random.normal(0, 0.5)
    return round(row["temperature"] - offset + noise, 1)

df["soil_temperature_c"] = df.apply(generate_soil_temperature, axis=1)


In [7]:
def generate_soil_moisture(row):
    factor = SOIL_MOISTURE_FACTOR.get(row["label"], SOIL_MOISTURE_FACTOR["Default"])
    
    base = (0.6 * row["rainfall"] + 0.4 * row["humidity"]) / 2
    noise = np.random.normal(0, 3)

    moisture = base * factor + noise
    return int(np.clip(moisture, 10, 100))

df["soil_moisture_pct"] = df.apply(generate_soil_moisture, axis=1)


In [8]:
df["soil_temperature"] = df["soil_temperature_c"].astype(str) + "°C"
df["soil_moisture"] = df["soil_moisture_pct"].astype(str) + "%"


In [9]:
df[
    [
        "temperature",
        "soil_temperature",
        "humidity",
        "rainfall",
        "soil_moisture",
        "label"
    ]
].head(10)


,temperature,soil_temperature,humidity,rainfall,soil_moisture,label
0,20.879744,17.6°C,82.002744,202.935536,82%,rice
1,21.770462,18.2°C,80.319644,226.655537,83%,rice
2,23.004459,19.8°C,82.320763,263.964248,97%,rice
3,26.491096,23.8°C,80.158363,242.864034,85%,rice
4,20.130175,16.5°C,81.604873,262.717340,92%,rice
5,23.058049,19.4°C,83.370118,251.055000,93%,rice
6,22.708838,20.0°C,82.639414,271.324860,100%,rice
7,20.277744,17.2°C,82.894086,241.974195,90%,rice
8,24.515881,20.8°C,83.535216,230.446236,84%,rice
9,23.223974,20.0°C,83.033227,221.209196,82%,rice


In [10]:
presentation_df = df.drop(columns=["soil_temperature_c", "soil_moisture_pct"])


In [11]:
output_csv = "Crop_recommendation_with_soil_features.csv"

df.to_csv(output_csv, index=False)

print("Output saved:", output_csv)
print("Final shape:", df.shape)


Output saved: Crop_recommendation_with_soil_features.csv
Final shape: (2200, 12)
